# Versi 2 — Parafrase Pertanyaan & Split Dataset (Ollama di Colab)

**Kenapa ada versi 2?** Model versi 1 sudah meniru gaya jawaban, tetapi belum hafal fakta (misal IPS 2,8 → dijawab 15 SKS, seharusnya 20 SKS). Setiap fakta hanya muncul sekali di data training, dan test set berisi fakta yang tidak pernah diajarkan.

**Perbaikan di notebook ini:**
1. Setiap fakta (607 QA hasil review) dibuatkan **3 variasi pertanyaan** dengan **jawaban yang sama persis** → ±2.400 contoh.
2. **Split per fakta:** semua fakta masuk train; val & test berisi **variasi kalimat yang tidak ikut dilatih** dari fakta yang sudah diajarkan → test mengukur apakah model benar-benar hafal fakta.

**Sebelum mulai:** Runtime **T4 GPU**, upload `data/processed/qa_bersih.jsonl` ke `MyDrive/proyek-llm/data/processed/`.

**Output (ke Drive `data/processed/`):** `parafrase_v2.jsonl`, `train_v2.jsonl`, `val_v2.jsonl`, `test_v2.jsonl`. Proses bisa dilanjutkan bila terputus.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR  = '/content/drive/MyDrive/proyek-llm/data/processed'
QA_PATH   = f'{DATA_DIR}/qa_bersih.jsonl'
PARA_PATH = f'{DATA_DIR}/parafrase_v2.jsonl'
assert os.path.exists(QA_PATH), f'Upload dulu qa_bersih.jsonl ke {QA_PATH}'
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install & jalankan Ollama

In [ ]:
%%capture
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, requests
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get('http://127.0.0.1:11434/api/version', timeout=2); print('Ollama server jalan'); break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('Ollama gagal start, cek /content/ollama.log')

In [ ]:
MODEL = 'qwen2.5:14b'   # ganti ke 'qwen2.5:7b' kalau ingin lebih cepat
!ollama pull {MODEL}

## 3. Prompt parafrase

In [ ]:
import json, re

PROMPT = '''Kamu membantu membuat variasi pertanyaan untuk dataset chatbot layanan akademik Universitas Telkom.

Untuk SETIAP item di bawah, buat 3 pertanyaan BARU yang maknanya SAMA PERSIS dengan pertanyaan asli,
sehingga jawabannya tetap jawaban yang tertulis (jangan ubah jawaban).
Aturan:
- Variasikan gaya: satu formal, satu santai (boleh pakai "kak", "gimana", "boleh gak", "nih"), satu singkat/langsung.
- Pertahankan semua angka, jenjang (sarjana/magister/doktor/diploma), dan istilah penting dari pertanyaan asli.
- Jangan menambah syarat, angka, atau informasi baru. Jangan menyalin pertanyaan asli.
- Gunakan bahasa Indonesia.

ITEM:
{items}

Keluarkan HANYA JSON dengan format:
{{"hasil": [{{"id": <id>, "parafrase": ["...", "...", "..."]}}]}}'''

def norm(s):
    return re.sub(r'[^a-z0-9 ]', '', s.lower()).strip()

def parafrase_batch(batch, coba=3):
    items = '\n'.join(f'- id={r["id"]} | pertanyaan: {r["pertanyaan"]} | jawaban: {r["jawaban"]}' for r in batch)
    for k in range(coba):
        r = requests.post('http://127.0.0.1:11434/api/chat', timeout=900, json={
            'model': MODEL, 'stream': False, 'format': 'json',
            'messages': [{'role': 'user', 'content': PROMPT.format(items=items)}],
            'options': {'temperature': 0.8, 'num_ctx': 8192}})
        try:
            r.raise_for_status()
            hasil = {int(h['id']): h['parafrase'] for h in json.loads(r.json()['message']['content'])['hasil']}
        except Exception as e:
            print(f'   gagal ({e}), coba lagi {k + 1}/{coba}'); continue
        out = {}
        for row in batch:
            asli = norm(row['pertanyaan'])
            ps, seen = [], {asli}
            for p in hasil.get(row['id'], []):
                if isinstance(p, str) and 8 <= len(p.strip()) <= 250 and norm(p) not in seen:
                    seen.add(norm(p)); ps.append(p.strip())
            out[row['id']] = ps[:3]
        return out
    return {}

fakta = [json.loads(l) for l in open(QA_PATH, encoding='utf-8')]
print(len(fakta), 'fakta')

## 4. Uji 1 batch dulu
Cek apakah variasi pertanyaan masih bermakna sama.

In [ ]:
contoh = parafrase_batch(fakta[100:104])
for row in fakta[100:104]:
    print('ASLI:', row['pertanyaan'])
    for p in contoh.get(row['id'], []): print('   ->', p)
    print('JAWAB:', row['jawaban'][:120], '\n')

## 5. Parafrase semua fakta (bisa dilanjutkan bila terputus)

In [ ]:
BATCH = 8
sudah = {}
if os.path.exists(PARA_PATH):
    for l in open(PARA_PATH, encoding='utf-8'):
        d = json.loads(l); sudah[d['id']] = d['parafrase']
print('Sudah ada:', len(sudah))

sisa = [r for r in fakta if r['id'] not in sudah or len(sudah[r['id']]) < 2]
t0 = time.time()
with open(PARA_PATH, 'a', encoding='utf-8') as fo:
    for i in range(0, len(sisa), BATCH):
        batch = sisa[i:i + BATCH]
        hasil = parafrase_batch(batch)
        for row in batch:
            ps = hasil.get(row['id'], [])
            if ps:
                sudah[row['id']] = ps
                fo.write(json.dumps({'id': row['id'], 'parafrase': ps}, ensure_ascii=False) + '\n')
        fo.flush()
        print(f'[{min(i + BATCH, len(sisa))}/{len(sisa)}] {time.time() - t0:.0f} s')
print('Fakta dengan parafrase:', sum(1 for r in fakta if len(sudah.get(r['id'], [])) >= 2), '/', len(fakta))

## 6. Split per fakta → train_v2 / val_v2 / test_v2

In [ ]:
import random

# HARUS sama persis dengan scripts/config.py dan deploy/Modelfile
SYSTEM_PROMPT = (
    "Kamu adalah asisten layanan informasi akademik Universitas Telkom. "
    "Jawab pertanyaan mahasiswa tentang peraturan dan regulasi akademik kampus "
    "dengan bahasa Indonesia yang sopan, singkat, dan jelas. "
    "Jika informasinya tidak ada dalam peraturan, katakan bahwa kamu tidak tahu "
    "dan sarankan menghubungi bagian akademik."
)

def contoh_chat(q, r, jenis):
    return {'messages': [{'role': 'system', 'content': SYSTEM_PROMPT},
                         {'role': 'user', 'content': q},
                         {'role': 'assistant', 'content': r['jawaban']}],
            'fakta_id': r['id'], 'jenis': jenis, 'kategori': r['kategori'], 'sumber': r['sumber']}

rng = random.Random(42)
ids = [r['id'] for r in fakta if len(sudah.get(r['id'], [])) >= 2]
rng.shuffle(ids)
n = len(ids)
test_ids = set(ids[:int(n * 0.12)])               # ±12% fakta: 1 variasi disimpan untuk test
val_ids  = set(ids[int(n * 0.12):int(n * 0.22)])  # ±10% fakta: 1 variasi disimpan untuk val

train, val, test = [], [], []
for r in fakta:
    ps = list(sudah.get(r['id'], []))
    train.append(contoh_chat(r['pertanyaan'], r, 'asli'))       # semua fakta asli masuk train
    if r['id'] in test_ids:
        test.append(contoh_chat(ps.pop(), r, 'parafrase_test'))
    elif r['id'] in val_ids:
        val.append(contoh_chat(ps.pop(), r, 'parafrase_val'))
    train += [contoh_chat(p, r, 'parafrase') for p in ps]
rng.shuffle(train)

for nama, data in (('train_v2', train), ('val_v2', val), ('test_v2', test)):
    with open(f'{DATA_DIR}/{nama}.jsonl', 'w', encoding='utf-8') as f:
        for d in data: f.write(json.dumps(d, ensure_ascii=False) + '\n')
print(f'train={len(train)}  val={len(val)}  test={len(test)}  (fakta={len(fakta)})')

## 7. Contoh hasil

In [ ]:
for d in test[:5]:
    print('Q:', d['messages'][1]['content']); print('A:', d['messages'][2]['content'][:150], '\n')

## Selanjutnya
1. **Runtime → Disconnect and delete runtime** (supaya GPU kosong untuk training).
2. Buka `notebooks/finetune_v2_colab.ipynb` di Colab dan jalankan.
3. (Untuk laporan) download `train_v2.jsonl`, `val_v2.jsonl`, `test_v2.jsonl`, `parafrase_v2.jsonl` ke `D:\proyek-llm\data\processed\`.